# 01. Nemotron 3 Nano Omni 기초

목표: 옴니모달 모델의 입력 modality와 token budget을 작은 계산으로 이해한다.

실행 방법:
1. Jupyter Notebook 또는 VS Code에서 이 파일을 연다.
2. 위에서 아래로 셀을 실행한다.
3. 이 노트북은 Python 표준 라이브러리만 사용한다.

이 실습은 공식 모델을 실행하지 않는다. NVIDIA 보고서와 DebuggerCafe 글의 구조를 바탕으로 token 수와 설계 선택을 학습하기 위한 축소 예제다.

In [ ]:
from dataclasses import dataclass
from math import ceil


@dataclass
class ImageInput:
    width: int
    height: int


@dataclass
class VideoInput:
    width: int
    height: int
    frames: int


@dataclass
class AudioInput:
    seconds: float

## 1. 이미지 token 수 추정

보고서는 이미지가 16x16 patch로 분해되고, visual token 수가 대략 1,024에서 13,312 사이로 제한된다고 설명한다. 여기서는 원본 비율을 보존한다는 dynamic resolution의 직관만 계산한다.

In [ ]:
def image_patch_tokens(image, patch_size=16, min_tokens=1024, max_tokens=13312):
    """이미지 해상도에서 16x16 patch token 수를 대략 계산한다.

    실제 모델은 dynamic resolution과 pixel shuffle을 함께 쓰므로 이 값은 근사치다.
    핵심은 해상도가 커질수록 decoder가 처리할 visual token 비용도 커진다는 점이다.
    """
    patches_w = ceil(image.width / patch_size)
    patches_h = ceil(image.height / patch_size)
    raw_tokens = patches_w * patches_h
    clipped_tokens = max(min_tokens, min(raw_tokens, max_tokens))
    return {
        "raw_tokens": raw_tokens,
        "clipped_tokens": clipped_tokens,
        "aspect_ratio": round(image.width / image.height, 3),
    }


images = [ImageInput(512, 512), ImageInput(1280, 720), ImageInput(1840, 1840), ImageInput(3000, 1200)]
for image in images:
    print(image, image_patch_tokens(image))

## 2. Pixel shuffle downsampling의 직관

보고서는 projection 전에 pixel shuffle로 4x downsampling을 적용한다고 설명한다. 아래 계산은 token 수가 4분의 1 수준으로 줄어드는 효과를 단순화해 보여준다.

In [ ]:
def apply_pixel_shuffle_reduction(tokens, reduction=4):
    """다운샘플링 후 decoder에 전달될 token 수를 단순 추정한다."""
    return ceil(tokens / reduction)


for image in images:
    tokens = image_patch_tokens(image)["clipped_tokens"]
    print(f"{image.width}x{image.height}: before={tokens}, after~={apply_pixel_shuffle_reduction(tokens)}")

## 3. 비디오 token 수 추정

비디오는 frame 수만큼 이미지 token 비용이 반복된다. Nemotron 3 Nano Omni는 Conv3D patch embedder로 두 frame을 하나처럼 압축해 temporal token을 약 2배 줄인다.

In [ ]:
def video_tokens(video):
    per_frame = image_patch_tokens(ImageInput(video.width, video.height))["clipped_tokens"]
    after_pixel_shuffle = apply_pixel_shuffle_reduction(per_frame)
    temporal_units = ceil(video.frames / 2)  # Conv3D가 두 frame을 하나의 temporal unit으로 압축한다고 단순화한다.
    return {
        "per_frame_tokens_after_visual_reduction": after_pixel_shuffle,
        "temporal_units_after_conv3d": temporal_units,
        "total_video_tokens": after_pixel_shuffle * temporal_units,
    }


videos = [VideoInput(1280, 720, 32), VideoInput(1280, 720, 128), VideoInput(1920, 1080, 128)]
for video in videos:
    print(video, video_tokens(video))

## 4. 오디오 token 수 추정

보고서는 오디오가 초당 약 12.5 token으로 표현된다고 설명한다. 긴 오디오는 context window를 빠르게 소비하므로, video+audio 입력에서는 예산 계산이 중요하다.

In [ ]:
def audio_tokens(audio, tokens_per_second=12.5):
    return ceil(audio.seconds * tokens_per_second)


for seconds in [10, 60, 20 * 60, 60 * 60, 5 * 60 * 60]:
    print(f"{seconds:>6} sec -> {audio_tokens(AudioInput(seconds))} audio tokens")

## 5. 통합 context budget 계산

공식 모델 카드 기준 최대 context는 256K token이다. 아래 함수는 text, image, video, audio가 합쳐질 때 예산을 넘는지 확인한다.

In [ ]:
MAX_CONTEXT = 256_000


def multimodal_budget(text_tokens=0, images=None, videos=None, audios=None):
    images = images or []
    videos = videos or []
    audios = audios or []

    image_total = sum(apply_pixel_shuffle_reduction(image_patch_tokens(image)["clipped_tokens"]) for image in images)
    video_total = sum(video_tokens(video)["total_video_tokens"] for video in videos)
    audio_total = sum(audio_tokens(audio) for audio in audios)
    total = text_tokens + image_total + video_total + audio_total
    return {
        "text_tokens": text_tokens,
        "image_tokens": image_total,
        "video_tokens": video_total,
        "audio_tokens": audio_total,
        "total": total,
        "fits_256k": total <= MAX_CONTEXT,
    }


request = multimodal_budget(
    text_tokens=1500,
    images=[ImageInput(1280, 720)],
    videos=[VideoInput(1280, 720, 128)],
    audios=[AudioInput(90)],
)
request

## 정리

- 옴니모달 모델은 여러 modality를 하나의 decoder context로 통합한다.
- 이미지/비디오 token은 해상도와 frame 수에 민감하다.
- audio token은 초당 token 비용으로 예산을 추정할 수 있다.
- dynamic resolution, pixel shuffle, Conv3D compression은 품질을 유지하면서 token 비용을 줄이기 위한 설계다.
- 다음 노트북에서는 NVIDIA API에 보낼 multimodal payload를 직접 조립한다.